# Reproducing Singh et al. (2026)

Light variant reproduction of the reference results from
*Maxitive Donsker–Varadhan Formulation for Possibilistic Variational Inference* (Singh et al., 2026). Compares AdamW, IVON, and uCBOpt (3 seeds each: 0, 1, 2) on a LeNet-5 variant (44,426 parameters) trained on Fashion-MNIST (100 epochs, linear warmup over 5 epochs + cosine decay), evaluated in-domain (Acc, NLL, ECE) and on EMNIST-Letters as OOD (FPR@95, AUROC).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MDV_reprod')
!pwd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/MDV_reprod


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import torch
import numpy as np
import pandas as pd

from src.model import LeNet
from src.data import get_fmnist_loaders, get_emnist_loader
from src.optimizers import build_optimizer
from src.train import train_model
from src.evaluate import indomain_metrics, ood_metrics

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = './data'
CHECKPOINT_DIR = './checkpoints'
CSV_PATH = './results/results_raw.csv'
SEEDS = [0,1,2]
EPOCHS = 100

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Hyperparameters from paper Appendix F.3 and reference implementation (100 epochs, no early stopping)
OPTIMIZER_CONFIGS = {
    'AdamW': dict(lr=1e-3, weight_decay=1e-2, beta1=0.9, beta2=0.999),
    'IVON':  dict(lr=0.2, weight_decay=2e-3, hess_init=0.5,
                  beta1=0.9, beta2=0.99999, ess=50000,
                  mc_samples=1, hess_approx='price', rescale_lr=True),
    'uCBOpt': dict(lr=1e-2, weight_decay=2e-3, hess_init=0.05,
                   cand_curvature=8e-6, beta1=0.9, beta2=0.99999, rescale_lr=False),
}

print('device:', DEVICE)

device: cuda


In [ ]:
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)

def train_optimizer(opt_name):
    """Train all seeds for one optimizer, save checkpoints and training history."""
    cfg = OPTIMIZER_CONFIGS[opt_name]
    for seed in SEEDS:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{opt_name}_seed{seed}.pt')
        if os.path.exists(ckpt_path):
            print(f'{opt_name} seed={seed} - checkpoint already exists, skipping.')
            continue
        print(f'Training {opt_name} seed={seed}...')
        set_seed(seed)
        train_loader, val_loader, _ = get_fmnist_loaders(
            DATA_DIR, batch_size=128, val_fraction=0.1, seed=seed)
        model = LeNet(num_classes=10, dropout=0.0).to(DEVICE)
        optimizer = build_optimizer(opt_name.lower(), model.parameters(), **cfg)
        model, history = train_model(
            model, optimizer, train_loader, val_loader,
            epochs=EPOCHS, warmup_epochs=5, device=DEVICE, verbose=True, patience=9999)
        torch.save({'model': model.state_dict(), 'opt': optimizer.state_dict()}, ckpt_path)
        print(f'  => saved to {ckpt_path}')

        # Save training history to CSV
        history_path = os.path.join(CHECKPOINT_DIR, f'{opt_name}_seed{seed}_history.csv')
        df_history = pd.DataFrame(history)
        df_history.insert(0, 'optimizer', opt_name)
        df_history.insert(1, 'seed', seed)
        df_history.to_csv(history_path, index=False)
        print(f'  => history saved to {history_path}')

In [ ]:
import os
os.makedirs('./data/EMNIST/raw', exist_ok=True)
!curl -L -o ./data/EMNIST/raw/gzip.zip https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip
!cd ./data/EMNIST/raw && unzip -o gzip.zip
!cp ./data/EMNIST/raw/gzip/emnist-letters-*.gz ./data/EMNIST/raw/
!gunzip -fk ./data/EMNIST/raw/emnist-letters-*.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  535M  100  535M    0     0  32.2M      0  0:00:16  0:00:16 --:--:-- 86.3M
Archive:  gzip.zip
  inflating: gzip/emnist-balanced-mapping.txt  
  inflating: gzip/emnist-balanced-test-images-idx3-ubyte.gz  
 extracting: gzip/emnist-balanced-test-labels-idx1-ubyte.gz  
  inflating: gzip/emnist-balanced-train-images-idx3-ubyte.gz  
  inflating: gzip/emnist-balanced-train-labels-idx1-ubyte.gz  
  inflating: gzip/emnist-byclass-mapping.txt  
  inflating: gzip/emnist-byclass-test-images-idx3-ubyte.gz  
  inflating: gzip/emnist-byclass-test-labels-idx1-ubyte.gz  
  inflating: gzip/emnist-byclass-train-images-idx3-ubyte.gz  
  inflating: gzip/emnist-byclass-train-labels-idx1-ubyte.gz  
  inflating: gzip/emnist-bymerge-mapping.txt  
  inflating: gzip/emnist-bymerge-test-images-idx3-ubyte.gz  
  inflating: gzip/emnist-bymerge-test-labels-

In [ ]:
train_optimizer('AdamW')

AdamW seed=0 - checkpoint already exists, skipping.
AdamW seed=1 - checkpoint already exists, skipping.
AdamW seed=2 - checkpoint already exists, skipping.


In [ ]:
train_optimizer('IVON')

IVON seed=0 - checkpoint already exists, skipping.
IVON seed=1 - checkpoint already exists, skipping.
IVON seed=2 - checkpoint already exists, skipping.


In [ ]:
train_optimizer('uCBOpt')

uCBOpt seed=0 - checkpoint already exists, skipping.
uCBOpt seed=1 - checkpoint already exists, skipping.
uCBOpt seed=2 - checkpoint already exists, skipping.


## Evaluation
Load saved checkpoints and compute in-domain + OOD metrics.

In [ ]:
def evaluate_optimizer(opt_name):
    """Load checkpoints and compute metrics for all seeds."""
    cfg = OPTIMIZER_CONFIGS[opt_name]
    _, _, test_loader = get_fmnist_loaders(DATA_DIR, batch_size=256, val_fraction=0.1, seed=0)
    ood_loader = get_emnist_loader(DATA_DIR)

    for seed in SEEDS:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{opt_name}_seed{seed}.pt')
        if not os.path.exists(ckpt_path):
            print(f'{opt_name} seed={seed} - checkpoint not found, skipping.')
            continue
        print(f'Evaluating {opt_name} seed={seed}...')
        model = LeNet(num_classes=10, dropout=0.0).to(DEVICE)
        optimizer = build_optimizer(opt_name.lower(), model.parameters(), **cfg)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['opt'])
        model.eval()

        metrics = {}
        metrics.update(indomain_metrics(model, test_loader, DEVICE, optimizer=optimizer))
        metrics.update(ood_metrics(model, test_loader, ood_loader, DEVICE, optimizer=optimizer))

        # Save to CSV
        row = {'optimizer': opt_name, 'seed': seed}
        row.update(metrics)
        df_new = pd.DataFrame([row])
        if os.path.exists(CSV_PATH):
            df_old = pd.read_csv(CSV_PATH)
            df_old = df_old[~((df_old['optimizer'] == opt_name) & (df_old['seed'] == seed))]
            df_final = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_final = df_new
        df_final.to_csv(CSV_PATH, index=False)
        print(metrics)

In [ ]:
evaluate_optimizer('AdamW')

Evaluating AdamW seed=0...
{'Acc': 0.8921, 'NLL': 0.30400383472442627, 'ECE': 0.01628897438198327, 'FPR@95': 0.33757198158169965, 'AUROC': 0.7006941706730769}
Evaluating AdamW seed=1...
{'Acc': 0.8963, 'NLL': 0.29297274351119995, 'ECE': 0.019786119815707194, 'FPR@95': 0.3277132314105708, 'AUROC': 0.7276658028846155}
Evaluating AdamW seed=2...
{'Acc': 0.8991, 'NLL': 0.2919714152812958, 'ECE': 0.030316805270314207, 'FPR@95': 0.3390566742435804, 'AUROC': 0.7374267740384616}


In [ ]:
evaluate_optimizer('IVON')

Evaluating IVON seed=0...
{'Acc': 0.9092, 'NLL': 0.2555139362812042, 'ECE': 0.01701153340041634, 'FPR@95': 0.23394213802192804, 'AUROC': 0.8256957932692307}
Evaluating IVON seed=1...
{'Acc': 0.9079, 'NLL': 0.26172441244125366, 'ECE': 0.02108121542334558, 'FPR@95': 0.25470184131872864, 'AUROC': 0.7968767716346152}
Evaluating IVON seed=2...
{'Acc': 0.9127, 'NLL': 0.2576974630355835, 'ECE': 0.017455115559697146, 'FPR@95': 0.23634476609157268, 'AUROC': 0.8133329158653846}


In [ ]:
evaluate_optimizer('uCBOpt')

Evaluating uCBOpt seed=0...
{'Acc': 0.9118, 'NLL': 0.2701635956764221, 'ECE': 0.024410168808698667, 'FPR@95': 0.2577791473863893, 'AUROC': 0.8016434374999999}
Evaluating uCBOpt seed=1...
{'Acc': 0.9065, 'NLL': 0.2683037221431732, 'ECE': 0.023212940095365044, 'FPR@95': 0.2618752142874635, 'AUROC': 0.7936333413461538}
Evaluating uCBOpt seed=2...
{'Acc': 0.9096, 'NLL': 0.27658501267433167, 'ECE': 0.02642125840485094, 'FPR@95': 0.26999044120024573, 'AUROC': 0.7887395168269231}


In [ ]:
df_raw = pd.read_csv('./results/results_raw.csv')

results = {name: [] for name in OPTIMIZER_CONFIGS}
for _, row in df_raw.iterrows():
    opt = row['optimizer']
    metrics = {k: row[k] for k in ['Acc', 'NLL', 'ECE', 'FPR@95', 'AUROC']}
    results[opt].append(metrics)

METRIC_COLS = ['Acc', 'NLL', 'ECE', 'FPR@95', 'AUROC']
rows = []
for opt_name, seed_results in results.items():
    row = {'Optimizer': opt_name}
    for m in METRIC_COLS:
        vals = np.array([r[m] for r in seed_results])
        row[m] = f'{vals.mean():.3f} +/- {vals.std():.3f}'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Optimizer')

# paper reference values (Table 1)
paper = pd.DataFrame([
    {'Optimizer': 'AdamW (paper)', 'Acc': '0.900 +/- 0.005', 'NLL': '0.294 +/- 0.007',
     'ECE': '0.026 +/- 0.004', 'FPR@95': '0.321 +/- 0.008', 'AUROC': '0.745 +/- 0.009'},
    {'Optimizer': 'IVON (paper)', 'Acc': '0.911 +/- 0.003', 'NLL': '0.258 +/- 0.001',
     'ECE': '0.013 +/- 0.004', 'FPR@95': '0.249 +/- 0.024', 'AUROC': '0.804 +/- 0.030'},
    {'Optimizer': 'uCBOpt (paper)', 'Acc': '0.907 +/- 0.002', 'NLL': '0.270 +/- 0.002',
     'ECE': '0.023 +/- 0.000', 'FPR@95': '0.245 +/- 0.010', 'AUROC': '0.810 +/- 0.012'},
]).set_index('Optimizer')

combined = pd.concat([df, paper])
print(combined.to_string())

combined.to_csv('./results/results_summary.csv')
rows_numeric = []
for opt_name, seed_results in results.items():
    for r in seed_results:
        row = {'Optimizer': opt_name}
        row.update(r)
        rows_numeric.append(row)
pd.DataFrame(rows_numeric).to_csv('./results/results_numeric.csv', index=False)


                            Acc              NLL              ECE           FPR@95            AUROC
Optimizer                                                                                          
AdamW           0.896 +/- 0.003  0.296 +/- 0.005  0.022 +/- 0.006  0.335 +/- 0.005  0.722 +/- 0.016
IVON            0.910 +/- 0.002  0.258 +/- 0.003  0.019 +/- 0.001  0.241 +/- 0.009  0.812 +/- 0.012
uCBOpt          0.909 +/- 0.002  0.272 +/- 0.004  0.025 +/- 0.001  0.263 +/- 0.005  0.795 +/- 0.005
AdamW (paper)   0.900 +/- 0.005  0.294 +/- 0.007  0.026 +/- 0.004  0.321 +/- 0.008  0.745 +/- 0.009
IVON (paper)    0.911 +/- 0.003  0.258 +/- 0.001  0.013 +/- 0.004  0.249 +/- 0.024  0.804 +/- 0.030
uCBOpt (paper)  0.907 +/- 0.002  0.270 +/- 0.002  0.023 +/- 0.000  0.245 +/- 0.010  0.810 +/- 0.012
